# Data Preprocessing

In this notebook we preprocess the data for each catchment in the sample to filter for basic quality and completeness criteria.  We take the resulting sample of daily mean timeseries and compute PMFs by discrete bin counting and kernel density estimation methods.  

Then for each set of PMFs, we compute the Kullback-Leibler Divergence (KLDs) between each pair of catchments in the sample.  The resulting pairwise KLD matrix is used as input to the divergence prediction model to test how the density estimation procedure affects the information in the distributions that can be learned by the gradient boosting model in subsequent notebooks.

The result of these pre-processing steps are provided in the open data repository.

In [1]:
from multiprocessing import Pool
from functools import partial

import os, sys
import pandas as pd
import geopandas as gpd
import json
import numpy as np
import xarray as xr
from shapely import Point

from pathlib import Path
from bokeh.plotting import figure, show, gridplot
from bokeh.io import output_notebook
import numpy as np
output_notebook()

from scipy.stats import linregress

# Add repo root to path
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import Config

from utils.kde_estimator import KDEEstimator
import utils.data_processing_functions as dpf

BASE_DIR = Path(os.getcwd())

# update this to the path where you stored `HYSETS_2023_update_QC_stations.nc`
HYSETS_DIR = Path('/home/danbot/code/common_data/HYSETS')


Loading BokehJS ...

## Import HYSETS catchment attributes

In [2]:
# import the HYSETS attributes data
hysets_df = pd.read_csv(HYSETS_DIR / 'HYSETS_watershed_properties.txt', sep=';')
ws_id_dict = hysets_df.set_index('Official_ID')['Watershed_ID'].to_dict()
da_dict = hysets_df.set_index('Official_ID')['Drainage_Area_km2'].to_dict()
official_id_dict = {row['Official_ID']: row['Watershed_ID'] for _, row in hysets_df.iterrows()}

### Import the pre-filtered stations within the study region

In [3]:
# import the BCUB (study) region boundary
rev_date = '20260203'
fpath = BASE_DIR / 'data' / f'Watershed_descriptors_{rev_date}.csv'
bcub_df = pd.read_csv(fpath)
bcub_df['official_id'] = bcub_df['official_id'].astype(str)
# map the Hysets watershed IDs to the BCUB watershed IDs
# create a dict to map HYSETS watershed IDs to the Official station IDs
bcub_df['watershedID'] = bcub_df['official_id'].apply(lambda x: official_id_dict.get(x, None))
bcub_df = bcub_df[~bcub_df['official_id'].isin(Config.EXCLUDED_STATIONS)].reset_index(drop=True)
da_dict = bcub_df.set_index('official_id')['drainage_area_km2'].to_dict()
stn_pts = [Point(xy) for xy in zip(bcub_df['centroid_lon_deg_e'], bcub_df['centroid_lat_deg_n'])]
bcub_gdf = gpd.GeoDataFrame(bcub_df, geometry=stn_pts, crs='EPSG:4326')
bcub_gdf = bcub_gdf.to_crs(epsg=3005)  # project to BC Albers for distance calculations
print(f'   Found {len(bcub_df)} catchments in the BCUB region with runoff statistics.')

   Found 1300 catchments in the BCUB region with runoff statistics.


## Import streamflow timeseries


```{note}
At the top of `data_processing_functions.py`, update the `STREAMFLOW_DIR` variable to match where the HYSETS streamflow time series are stored.  
```


In [4]:
# Load dataset
# streamflow = xr.open_dataset(HYSETS_DIR / 'HYSETS_2023_update_QC_stations.nc')

# # Promote 'watershedID' to a coordinate on 'watershed'
# streamflow = streamflow.assign_coords(watershedID=("watershed", streamflow["watershedID"].data))

# # Set 'watershedID' as index
# streamflow = streamflow.set_index(watershed="watershedID")

# # Select only watershedIDs present in bcub_df
# valid_ids = [int(wid) for wid in bcub_df['watershedID'].values if wid in streamflow.watershed.values]
# ds = streamflow.sel(watershed=valid_ids)

USGS station IDs are integers but they are stored in the dataset with the unfortunate characteristic that different stations can have identifiers that are substrings of each other.  We have to add a few extra lines of code to ensure we get the correct file.

In [5]:
# Confirm all watershed IDs exist in ds
# bcub_ws_ids = bcub_df['watershedID'].values
# ds_ids = ds.watershed.values  # After selection via sel(watershed=valid_ids)

# missing = [wid for wid in bcub_ws_ids if wid not in ds_ids]
# n_total = len(bcub_ws_ids)
# n_missing = len(missing)

# assert n_missing == 0, f"{n_missing} / {n_total} watershedIDs missing from dataset. First few missing: {missing[:5]}"

In [6]:
# # compute distances between all station (catchment centroids)
dist_dict = {}
dist_cache_path = BASE_DIR / 'data' / f'station_distances_{rev_date}.json'
if dist_cache_path.exists():
    with open(dist_cache_path, 'r') as f:
        dist_dict = json.load(f)
else:
    if not dist_dict:
        for stn in bcub_gdf['official_id'].values:
            point = bcub_gdf[bcub_gdf['official_id'] == stn].geometry.values[0]
            assert bcub_gdf.crs == 'EPSG:3005', 'BCUB GeoDataFrame CRS is not EPSG:3005, must be in projected CRS to compute distances correctly.'
            bcub_gdf.loc[bcub_gdf['official_id'] == stn, 'x'] = point.x
            bcub_gdf.loc[bcub_gdf['official_id'] == stn, 'y'] = point.y
            dist_dict[stn] = {}
            for stn2 in bcub_gdf['official_id'].values:
                point2 = bcub_gdf[bcub_gdf['official_id'] == stn2].geometry.values[0]
                dist_dict[stn][stn2] = point.distance(point2) / 1000.0  # convert to km

            if len(dist_dict.keys()) % 100 == 0:
                print(f'   Processed distances for {100*len(dist_dict.keys())/len(bcub_gdf):.0f}% of the station sample.')

    with open(dist_cache_path, 'w') as f:
        json.dump(dist_dict, f)


In [8]:
class ReferenceDistribution:
    def __init__(self, **kwargs):

        for k, v in kwargs.items():
            setattr(self, k, v)

        self._initialize_station()
        self._digitize_uar_series()


    def _initialize_station(self):
        self.da = self.da_dict[self.stn]
        self.df, self.zero_flow_flag = retrieve_and_preprocess_timeseries_discharge(stn)
        self.df['uar'] = 1000 * self.df['discharge'] / self.da
        self.complete_year_data = self.complete_year_dict[self.stn]
        self.complete_hyd_years = self.complete_year_data['hyd_years']
        self.complete_cal_years = self.complete_year_data['cal_years']
        # filter to only complete years and drop nan values
        self.hyd_df = self.df[self.df.index.year.isin(self.complete_hyd_years)].copy()
        self.cal_df = self.df[self.df.index.year.isin(self.complete_cal_years)].copy()
        self.hyd_df = self.hyd_df.dropna(subset=['uar'])
        self.cal_df = self.cal_df.dropna(subset=['uar'])


    def _digitize_uar_series(self):
        # digitize the uar series
        lin_edges_extended = np.exp(self.log_edges_extended)
        self.minimum_uar_threshold = float(1000.0 * self.zero_equiv_flow_threshold / self.da)
        self.hyd_df['uar_bin'] = np.digitize(self.hyd_df['uar'], lin_edges_extended, right=False) - 1 # hydrologic year data
        self.cal_df['uar_bin'] = np.digitize(self.cal_df['uar'], lin_edges_extended, right=False) - 1 # calendar year data
        # determine the bin index corresponding to the minimum measurable threshold
        self.zero_bin_index = max(0, np.digitize(self.minimum_uar_threshold, lin_edges_extended, right=False) - 1)
        
        # map the quantized bin values back to the series (midpoint in log space)
        self.lin_x_extended = np.exp(0.5 *(self.log_edges_extended[1:] + self.log_edges_extended[:-1]))
        self.hyd_df['uar_discrete'] = self.lin_x_extended[self.hyd_df['uar_bin'].clip(0, np.inf)]
        # clip the bin indices to valid range
        assert self.hyd_df['uar_bin'].max() < len(self.lin_x_extended), f"uar_bin index out of range. {self.hyd_df['uar_bin'].max()} >= {len(self.lin_x_extended)}"
        
        # handle bin values below the minimum measurable threshold
        self.hyd_df['uar_bin_adjusted'] = self.hyd_df['uar_bin'].copy()
        # handle values below the minimum measurable threshold
        self.hyd_df['uar_zero_adjusted'] = self.hyd_df['uar'].copy()
        if self.hyd_df['uar_bin'].min() < self.zero_bin_index and self.zero_bin_index > 0:
            # get the minimum log value
            # the discrete x value to the left of the bin containing the "minimum measurable value"
            min_uar = self.lin_x_extended[self.zero_bin_index - 1] 
            self.hyd_df.loc[self.hyd_df['uar_bin'] < 0, 'uar_discrete'] = np.float32(min_uar)            
            # adjust the uar bin where the bin index is smaller than the zero bin index            
            self.hyd_df.loc[self.hyd_df['uar_bin_adjusted'] < self.zero_bin_index, 'uar_bin_adjusted'] = 0
            # adjust the uar values below the minimum measurable threshold
            self.hyd_df.loc[self.hyd_df['uar_bin_adjusted'] < self.zero_bin_index, 'uar_zero_adjusted'] = np.float32(min_uar)


    def _D_bits_Q_to_Qlam(self, Q, lam):
        """Exact D_bits(Q || Q_lam) for Q_lam = (1-lam)Q + lam U."""
        Q = np.asarray(Q, dtype=float)
        Q = np.clip(Q / Q.sum(), 1e-300, 1.0)
        N = Q.size
        U = 1.0 / N
        Qlam = (1.0 - lam) * Q + lam * U
        return np.sum(Q * (np.log2(Q) - np.log2(Qlam)))
    
    
    def _compute_optimal_delta_limited_lambda(self, Q, maxit=100, tol=1e-6):
        """
        Largest λ ∈ [0,1] such that D_bits(Q || Q_λ) ≤ δ (exact, monotone bisection).
        """
        lo, hi = 0.0, 1.0
        if self._D_bits_Q_to_Qlam(Q, hi) <= self.delta:
            return hi
        for _ in range(maxit):
            mid = 0.5 * (lo + hi)
            if self._D_bits_Q_to_Qlam(Q, mid) <= self.delta:
                lo = mid
            else:
                hi = mid
            if hi - lo < tol:
                break
        return lo
    

    def _mix_with_uniform(self, Q: np.ndarray, lam: float) -> np.ndarray:
        """Shrink Q toward uniform with weight λ.  
        Applies a Dirichlet (uniform) prior to the distribution.
        lam = lambda = strength of belief in the prior
        """
        U = np.ones_like(Q) / len(Q)
        return (1.0 - lam) * Q + lam * U
    

    def _compute_adjusted_distribution_with_mixed_uniform(self, pmf):
        """
        Compute a mixture Q_mixed = (1 - alpha) * Q_kde + alpha * Q_uniform
        The mixture is limited by delta, the maximum allowed perturbation between
        Q_mixture and the original Q.
        Given delta, we can compute the largest allowable lambda,
        which represents the most noise added without overly influencing Q.
        """
        lam_exact = self._compute_optimal_delta_limited_lambda(pmf)
        pmf_mixed = self._mix_with_uniform(pmf, lam_exact)
        # pdf_mixed = pmf_mixed / self.log_w
        assert np.isclose(np.sum(pmf_mixed), 1.0), f'Mixed PMF does not sum to 1: {np.sum(pmf_mixed)}'
        pmf_mixed /= pmf_mixed.sum()

        # pdf_check = np.trapezoid(pdf_mixed, x=self.log_x)
        # pdf_mixed /= pdf_check

        return pmf_mixed
    
        
    
    def build_station_pmf(self):
        """
        Build the discretized and KDE smoothed PMFs for the daily uar timeseries.
        """

        # counts = np.histogram(log_uar, bins=self.pos_edges, density=False)[0].astype(np.int64, copy=False)
        unique_bin_idxs, bin_counts = np.unique(self.hyd_df['uar_bin_adjusted'].values, return_counts=True)

        # initialize the PMF 
        pmf = np.zeros(len(self.lin_x_extended))

        # assign the observation counts to the pmf by bin index
        pmf[unique_bin_idxs] = bin_counts.astype(int)

        # assert the counts match
        assert pmf.sum() == len(self.hyd_df), f"PMF counts {pmf.sum()} do not match number of observations {len(self.hyd_df)}"

        # normalize to PMF
        pmf /= pmf.sum()
        
        # KDE on the same grid given the observed uar values
        # with the smallest value adjusted to the minimum measurable threshold
        positive_values = self.hyd_df[self.hyd_df['uar'] >= self.minimum_uar_threshold]['uar'].values
        N_p = len(positive_values)
        N_n = len(self.hyd_df) - N_p
        pmf_kde_raw, _ = self.kde_estimator.compute(
            positive_values,
            self.drainage_area_km2
            )  # returns pmf over pos_edges intervals
        
        
        kde_counts = (pmf_kde_raw * N_p)

        assert len(kde_counts) == len(self.lin_x_extended), f"KDE counts length {len(kde_counts)} does not match PMF length {len(self.lin_x_extended)}"
        
        if N_n > 0:
            pmf_kde = np.zeros_like(kde_counts)
            if self.zero_bin_index == 0: 
                # the minimum measurable threshold is below the support, N_n are zero flows.
                # all zero flows go to the first bin and there is no lower bin mass to consider
                low_probability_mass = N_n 
            else:
                # compute the low probability mass from the KDE below the zero bin index
                low_probability_mass = N_n + kde_counts[:self.zero_bin_index].sum()
            
            # if zero_bin_index == 0, the zero index bin will be reassigned in the second step
            pmf_kde[self.zero_bin_index:] = kde_counts[self.zero_bin_index:]
            pmf_kde[0] = low_probability_mass
            # print(N_n, low_probability_mass, pmf_kde[0], self.zero_bin_index)
            
        else:
            assert N_p == len(self.hyd_df)
            pmf_kde = (pmf_kde_raw * len(self.hyd_df))

        count_match = int(pmf_kde.sum() - len(self.hyd_df))
            
        assert np.isclose(count_match, 0), f"PMF counts {pmf_kde.sum()} do not match number of observations {len(self.hyd_df)} after zero bin adjustment"
        pmf_kde /= pmf_kde.sum()  # renormalize to PMF
        assert np.isclose(pmf_kde.sum(), 1.0), f"KDE PMF does not sum to 1: {pmf_kde.sum()}"
        assert np.isclose(pmf.sum(), 1.0), f"Discrete PMF does not sum to 1: {pmf.sum()}"
        return pmf, pmf_kde


## Process Pairwise KL Divergence

Here we use previous methods from the FDC estimation study.

1. Assume 10 bits with bins equally (log) spaced over the global range of streamflow values in the dataset.
2. Apply a uniform prior mixture distribution to the "donor" catchment PMF to avoid zero probabilities.

In [9]:
def D_bits_Q_to_Qlam(Q, lam):
    """Exact D_bits(Q || Q_lam) for Q_lam = (1-lam)Q + lam U."""
    Q = np.asarray(Q, dtype=float)
    Q = np.clip(Q / Q.sum(), 1e-300, 1.0)
    N = Q.size
    U = 1.0 / N
    Qlam = (1.0 - lam) * Q + lam * U
    return np.sum(Q * (np.log2(Q) - np.log2(Qlam)))


def compute_optimal_delta_limited_lambda(Q, delta, maxit=100, tol=1e-6):
    """
    Largest λ ∈ [0,1] such that D_bits(Q || Q_λ) ≤ δ (exact, monotone bisection).
    """
    lo, hi = 0.0, 1.0
    if D_bits_Q_to_Qlam(Q, hi) <= delta:
        return hi
    for _ in range(maxit):
        mid = 0.5 * (lo + hi)
        if D_bits_Q_to_Qlam(Q, mid) <= delta:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return lo


def mix_with_uniform(Q: np.ndarray, lam: float) -> np.ndarray:
    """Shrink Q toward uniform with weight λ.  
    Applies a Dirichlet (uniform) prior to the distribution.
    lam = lambda = strength of belief in the prior
    """
    U = np.ones_like(Q) / len(Q)
    return (1.0 - lam) * Q + lam * U


def compute_adjusted_distribution_with_mixed_uniform(pmf, delta):
    """
    Compute a mixture Q_mixed = (1 - alpha) * Q_kde + alpha * Q_uniform
    The mixture is limited by delta, the maximum allowed perturbation between
    Q_mixture and the original Q.
    Given delta, we can compute the largest allowable lambda,
    which represents the most noise added without overly influencing Q.
    """
    lam_exact = compute_optimal_delta_limited_lambda(pmf, delta)
    pmf_mixed = mix_with_uniform(pmf, lam_exact)
    # pdf_mixed = pmf_mixed / self.log_w
    assert np.isclose(np.sum(pmf_mixed), 1.0), f'Mixed PMF does not sum to 1: {np.sum(pmf_mixed)}'
    pmf_mixed /= pmf_mixed.sum()

    # pdf_check = np.trapezoid(pdf_mixed, x=self.log_x)
    # pdf_mixed /= pdf_check

    return pmf_mixed

In [10]:
def pairwise_kld_long(df_P):
    """
    Compute all pairwise KL divergences D_KL(P || Q) in base-2 and
    return a long-form DataFrame with columns:
        - donor:  Q in D_KL(P || Q)
        - target: P in D_KL(P || Q)
        - dkl:    divergence value
    """
    df_Q = df_P.copy()  # when computing pairwise divergences within the same set, P and Q are the same
    P_cols = np.array(df_P.columns)
    Q_cols = np.array(df_Q.columns)
    P = df_P.to_numpy(float)  # shape (N, M_P)
    Q = df_Q.to_numpy(float)  # shape (N, M_Q)

    # log2(P) with mask on P > 0
    logP = np.zeros_like(P)
    maskP = P > 0
    logP[maskP] = np.log2(P[maskP])

    # h_i = sum_r P_r,i * log2 P_r,i  (entropy-like term, one per P-column)
    h = (P * logP).sum(axis=0)  # shape (M_P,)

    # log2(Q)
    logQ = np.log2(Q)  # shape (N, M_Q)

    # S_ij = sum_r P_r,i * log2 Q_r,j
    S = P.T @ logQ  # shape (M_P, M_Q)

    # D_ij = D_KL(P_i || Q_j) in base-2
    D = h[:, None] - S  # shape (M_P, M_Q)

    # Build long-form indices: each row is (P_i, Q_j)
    M_P, M_Q = D.shape
    p_idx = np.repeat(np.arange(M_P), M_Q)     # target index (P)
    q_idx = np.tile(np.arange(M_Q), M_P)       # donor index (Q)

    donor = Q_cols[q_idx].astype(str)       # q in D_KL(P||Q)
    target = P_cols[p_idx].astype(str)      # p in D_KL(P||Q)
    dkl_vals = D.reshape(-1)

    out = pd.DataFrame({
        "donor": donor,
        "target": target,
        "kld": dkl_vals,
    })
    # assert no donor == target
    # assert not np.any(out['donor'] == out['target']), "Donor and target should not be the same in pairwise KL divergence output."

    return out

In [11]:
# target_cols = ['mean_uar', 'sd_uar', 'mean_logx', 'sd_logx']
target_cols = ['dkl']
delta = 0.001

# set paths for catchment attributes and meteorological forcing data
baseline_distribution_folder = BASE_DIR / 'data' / 'baseline_distributions'

result_dict = {}
for bitrate in [8]:#[4, 5, 6, 7, 8, 9, 10, 11, 12]:
    out_folder = BASE_DIR / 'data' / 'kld_batches' / f'{bitrate}_bits'
    result_dict[bitrate] = {}
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)

    for density_estimator in ['kde']: # 'kde' or 'obs' (discrete bin counting)
        output_fpath = out_folder / f'pairwise_kld_{bitrate}_bits_{density_estimator}.csv'
        if os.path.exists(output_fpath):
            print(f'Pairwise KL divergence file already exists: {output_fpath}. Skipping computation.')
            result_dict[bitrate][density_estimator] = pd.read_csv(output_fpath, dtype={'donor': str, 'target': str, 'kld': float})
            continue

        print(f'Processing pairwise KL divergence for {bitrate}-bit baseline distributions using {density_estimator} regularization.')

        # context = FDCEstimationContext(**input_data)
        baseline_pmf_path = baseline_distribution_folder / f'{bitrate:02d}_bits' / f'pmf_kde_adaptive_mixture.csv'
        baseline_pmf_df = pd.read_csv(baseline_pmf_path)
        stn_id_cols = [c for c in baseline_pmf_df.columns if c not in ['log_x_uar']]
        baseline_pmf_df = baseline_pmf_df[stn_id_cols]
        
        # compute all pairwise KL divergences
        all_klds = pairwise_kld_long(baseline_pmf_df)
        all_klds = all_klds[all_klds['target'] != all_klds['donor']].reset_index(drop=True)
        # add the distance between
        all_klds['centroid_distance_km'] = all_klds.apply(lambda row: dist_dict[row['target']][row['donor']], axis=1)
        all_klds.to_csv(output_fpath, index=False)
        result_dict[bitrate][density_estimator] = all_klds
        msg = f'    {bitrate} bit Pairwise KL divergence results saved to {output_fpath}'
        print(msg)


Pairwise KL divergence file already exists: /home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/data/kld_batches/8_bits/pairwise_kld_8_bits_kde.csv. Skipping computation.


Check the effectiveness of the distance and attribute selection criteria.

1. For each target catchment, load the kNN results from the FDC estimation study.
2. For each catchment != target catchment, compute the KL divergence between the target and donor catchment PMFs.
3. Check that the 1NN result from the kNN results matches the direct KL divergence result.
4. Store the result of all pairswise comparisons for later analysis.


In [12]:
def min_pairwise_kld(df_P, df_Q):
    """
    Vectorized KL divergence minimization in base-2.
    """
    P_cols = np.array(df_P.columns)
    Q_cols = np.array(df_Q.columns)
    P = df_P.to_numpy(float)
    Q = df_Q.to_numpy(float)

    # log P with mask on P > 0
    logP = np.zeros_like(P)
    maskP = P > 0
    logP[maskP] = np.log2(P[maskP])

    h = (P * logP).sum(axis=0)          # (M_P,)
    logQ = np.log2(Q)
    S = P.T @ logQ                      # (M_P, M_Q)
    D = (h[:, None] - S)# / np.log(2.0)  # (M_P, M_Q)

    mask = np.eye(len(P_cols), dtype=bool) # mask i=j
    D_masked = np.where(mask, np.inf, D) # set j=j (diagonal to inf to avoid picking self-divergence)

    j2_idx = np.argmin(D_masked, axis=1)
    g_min = D[np.arange(D.shape[0]), j2_idx]  # use unmasked D for actual values

    result = {
        p: {
            "min_j2_idx": int(k),
            "min_j2_col": Q_cols[k],
            "min_kl": float(v),
        }
        for p, k, v in zip(P_cols, j2_idx, g_min)
    }

    return result

In [13]:
def nearest_pair_mixture_kl(df_P, df_Q):
    """
    For each column j1 in df_P, find the pair of distinct columns (j2, j3)
    in df_Q, with j2 != j3 and j1 not in {j2, j3}, that minimizes

        D_KL(P_j1 || 0.5 * (Q_j2 + Q_j3))     (in bits)

    Assumes:
    - df_P and df_Q have the same columns, same order (same station IDs).
    - Columns are valid pmfs (non-negative, sum ~ 1).
    - Any zero-handling for Q has already been done so log2(Q) is finite;
      mixtures of Q are then also > 0.
    """

    P_cols = np.array(df_P.columns)
    Q_cols = np.array(df_Q.columns)

    # Require same stations for sensible "exclude j1 from mixture"
    assert len(P_cols) == len(Q_cols), "df_P and df_Q must have same number of columns"
    assert np.all(P_cols == Q_cols), "df_P and df_Q columns must match and be aligned"

    P = df_P.to_numpy(float)   # shape (N, M)
    Q = df_Q.to_numpy(float)   # shape (N, M)
    N, M = P.shape

    # ---- KL ingredients for P -------------------------------------------------
    # log P with mask on P > 0 (P == 0 contributes 0 to KL)
    logP = np.zeros_like(P)
    maskP = P > 0
    logP[maskP] = np.log2(P[maskP])

    # h[j] = sum_i P_ij * log P_ij
    h = (P * logP).sum(axis=0)        # shape (M,)

    # ---- all unordered distinct pairs (j2 < j3) -------------------------------
    j2_idx, j3_idx = np.triu_indices(M, k=1)  # both shape (K,)
    K = j2_idx.size

    # Mixture Q_mix = 0.5 * (Q[:, j2] + Q[:, j3]), shape (N, K)
    Q2 = Q[:, j2_idx]
    Q3 = Q[:, j3_idx]
    Q_mix = 0.5 * (Q2 + Q3)

    # renormalize mixtures to ensure they sum to 1
    Q_mix /= Q_mix.sum(axis=0, keepdims=True)

    # log of mixture
    logQ_mix = np.log2(Q_mix)          # shape (N, K)

    # S[j1, k] = sum_i P_ij1 * log Q_mix_ik
    S = P.T @ logQ_mix                # shape (M, K)

    # KL in bits: D[j1, k] = h[j1] - S[j1, k]
    D = h[:, None] - S                # shape (M, K)

    # ---- mask out pairs that contain the target j1 ----------------------------
    # For each j1 (row), we want to exclude all k where j1 is either j2_idx[k] or j3_idx[k]
    rows = np.arange(M)[:, None]      # shape (M, 1)
    pair_mask = (rows == j2_idx[None, :]) | (rows == j3_idx[None, :])  # (M, K)

    # Set invalid candidates to +inf, so argmin ignores them
    D_masked = np.where(pair_mask, np.inf, D)

    # For each j1, find best pair index k*
    best_k = np.argmin(D_masked, axis=1)     # shape (M,)
    best_kl = D[np.arange(M), best_k]        # use unmasked D for actual values

    # Map best pair indices back to neighbour indices
    best_j2 = j2_idx[best_k]                 # shape (M,)
    best_j3 = j3_idx[best_k]                 # shape (M,)

    # ---- build result dict ----------------------------------------------------
    result = {}
    for j1, p_name in enumerate(P_cols):
        n2 = int(best_j2[j1])
        n3 = int(best_j3[j1])
        result[p_name] = {
            "neigh1_idx": n2,
            "neigh1_col": Q_cols[n2],
            "neigh2_idx": n3,
            "neigh2_col": Q_cols[n3],
            "min_kl": float(best_kl[j1]),
        }

    return result

In [14]:
# set paths for catchment attributes and meteorological forcing data
baseline_distribution_folder = BASE_DIR / 'data' / 'baseline_distributions'

min_kld_dict = {}
for bitrate in [8]:#[4, 5, 6, 7, 8, 9, 10, 11, 12]:
    out_folder = BASE_DIR / 'data' / 'kld_batches' / f'{bitrate}_bits'
    print(f'Processing minimum pairwise KL divergence for {bitrate}-bit baseline distributions.')
    min_kld_dict[bitrate] = {}
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)

    for density_estimator in ['kde', 'obs']: # 'kde' or 'obs' (discrete bin counting)
        output_fpath = out_folder / f'pairwise_kld_{bitrate}_bits_{density_estimator}.csv'
        baseline_pmf_path = baseline_distribution_folder / f'{bitrate:02d}_bits' / f'pmf_kde_adaptive_mixture.csv'
        baseline_pmf_df = pd.read_csv(baseline_pmf_path)
        stn_id_cols = [c for c in baseline_pmf_df.columns if c not in ['log_x_uar']]
        baseline_pmf_df = baseline_pmf_df[stn_id_cols]
        # baseline_pmfs_adjusted = baseline_pmf_df.apply(lambda col: compute_adjusted_distribution_with_mixed_uniform(col, delta), axis=0)
        # assert np.allclose(baseline_pmfs_adjusted.sum(), len(baseline_pmfs_adjusted.columns) * [1.0]), 'Adjusted PMFs do not sum to 1 per station'
        # find the minimum pairwise KL divergence for each station
        min_kls = min_pairwise_kld(baseline_pmf_df, baseline_pmf_df.copy())
        min_kld_dict[bitrate][density_estimator] = min_kls


Processing minimum pairwise KL divergence for 8-bit baseline distributions.


In [15]:
def retrieve_knn_result(stn, bitrate=8):
    knn_result_fname = f'{stn}_fdc_results.json'
    # knn_result_folder = RESULTS_DIR / f'fdc_estimation_results_{bitrate:02d}_bits' / 'knn'
    knn_result_folder = Path('/home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn')

    if not os.path.exists(knn_result_folder / knn_result_fname):
        print(f'    {knn_result_folder}/{knn_result_fname} not found, trying padded zero version...')
        # pad a zero on the front and try again
        knn_result_fname = f'0{stn}_fdc_results.json'
        if not os.path.exists(knn_result_folder / knn_result_fname):
            # print(f'    KNN results file not found for station {target_stn}, skipping...')
            return {}

    with open(knn_result_folder / knn_result_fname, 'r') as f:
        knn_result = json.load(f)
    return knn_result

In [16]:
selection_success = {}
exhaustive_best_kl = {i: [] for i in range(1, 11)}
test_bitrate = 8
min_kls = min_kld_dict[test_bitrate]['obs']
for stn, data in min_kls.items():
    # get the KNN results for this station
    knn_result = retrieve_knn_result(stn, bitrate=test_bitrate)
    
    selection_success[stn] = {}
    attr_combo_string = 'attribute_dist_ID2_freqEnsemble'
    dist_combo_string = 'spatial_dist_ID2_freqEnsemble'
    # attr_keys = [k for k in knn_result.keys() if k.startswith(f'{stn}_') and attr_combo_string in k]
    # dist_keys = [k for k in knn_result.keys() if k.startswith(f'{stn}_') and dist_combo_string in k]
    if not knn_result:
        # print(f'       KNN results not found for station {stn}, skipping...')
        continue
    for i in range(1, 11):
        k_attr = f'{stn}_{i}_NN_{attr_combo_string}'
        k_dist = f'{stn}_{i}_NN_{dist_combo_string}'
        selection_success[stn][i] = {}
        for l in ['distance', 'attribute']:
            # get the result according to the selection criterion and number of neighbors
            rdat = knn_result.get(k_attr if l == 'attribute' else k_dist, None)
            selected_nbr = rdat['nbrs']
            knn_kld = rdat['eval']['kld'] # the kNN result

            # compare to the minimum KL divergence found
            min_kl_station = min_kls[stn]['min_j2_col']
            min_kl_exhaustive = min_kls[stn]['min_kl']
            stn_info = (stn, l, selected_nbr, knn_kld, min_kl_station, min_kl_exhaustive)
            exhaustive_best_kl[i].append(stn_info)

    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn/05014500_fdc_results.json not found, trying padded zero version...
    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn/05017500_fdc_results.json not found, trying padded zero version...
    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn/08OA005_fdc_results.json not found, trying padded zero version...
    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn/12042800_fdc_results.json not found, trying padded zero version...
    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results/knn/12102140_fdc_results.json not found, trying padded zero version...
    /home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_

In [17]:
match_df = pd.DataFrame(exhaustive_best_kl[1], columns=['official_id', 'selection', 'knn_nbr', 'knn_kl', 'ex_nbr', 'ex_value'])
matched = match_df[match_df['knn_nbr'] == match_df['ex_nbr']].copy()
matched['ratio'] = matched['ex_value'] / matched['knn_kl']
print(len(matched), f'matches found for 1NN selection vs exhaustive search.')
print(matched.head())

128 matches found for 1NN selection vs exhaustive search.
   official_id  selection  knn_nbr    knn_kl   ex_nbr  ex_value     ratio
42     05CC001   distance  05CC007  0.115069  05CC007  0.113829  0.989223
43     05CC001  attribute  05CC007  0.115069  05CC007  0.113829  0.989223
47     05DA007  attribute  05DA009  0.078082  05DA009  0.077177  0.988412
48     05DA009   distance  05DA007  0.061075  05DA007  0.059851  0.979966
49     05DA009  attribute  05DA007  0.061075  05DA007  0.059851  0.979966


In [18]:

p = figure(title='KNN Neighbor Selection Matching Minimum KL Divergence Neighbor', width=600, height=400)
colors = ['red', 'black']
for i, selection in enumerate(['distance', 'attribute']):
    subset = matched[matched['selection'] == selection].copy()
    n_total = len(match_df)
    n_matched = len(subset[subset['knn_nbr'] == subset['ex_nbr']])
    match_rate = n_matched / n_total * 100.0
    print(f'Selection method: {selection}, Match rate: {n_matched}/{n_total} = {match_rate:.2f}%')
    p.scatter(subset['ex_value'], subset['knn_kl'], color=colors[i], alpha=0.6)

    slope, intercept, r_value, p_value, std_err = linregress(subset['ex_value'], subset['knn_kl'])
    x_vals = np.array([subset['ex_value'].min(), subset['ex_value'].max()])
    y_vals = intercept + slope * x_vals
    p.line(x_vals, y_vals, color=colors[i], line_width=2, 
           legend_label=f'{selection}: y={slope:.2f}x+{intercept:.2f}, r²={r_value**2:.2f}')

# make 1:1 line
x_vals = np.array([matched['ex_value'].min(), matched['ex_value'].max()])
p.line(x_vals, x_vals, color='black', line_width=2, line_dash='dotted', legend_label='1:1')
p.xaxis.axis_label = 'Minimum KL Divergence'
p.yaxis.axis_label = 'KNN Selection KL Divergence'
p.legend.location = 'top_left'
p = dpf.format_fig_fonts(p, font_size=14)
show(p)

Selection method: distance, Match rate: 52/1412 = 3.68%
Selection method: attribute, Match rate: 76/1412 = 5.38%


The point of the above comparison is to ensure that the methods are consistent and that the kNN search is effectively finding the minimum KL divergence catchment in the sample.  If this is not the case, then way the PMFs were computed between methods is likely not consistent.

In [19]:
p = figure(width=600, height=400, title=f'KL Divergence to Nearest Neighbor Baseline PMF (N={len(match_df)})', 
        x_axis_type='log')#, toolbar_location=None)

for i, selection in enumerate(['distance', 'attribute']):
    subset = match_df[match_df['selection'] == selection].copy()
    sorted_min_vals = np.sort(subset['ex_value'].values)
    sorted_knn_vals = np.sort(subset['knn_kl'].values)

    n_vals = len(sorted_knn_vals)
    assert len(sorted_knn_vals) == len(sorted_min_vals), 'Sorted values length mismatch.'
    
    ecdf_vals = np.arange(1, n_vals + 1) / n_vals

    # p.line(sorted_kl, ecdf_kl_vals, line_width=2, line_color='navy', legend_label='All Stations')
    p.line(sorted_min_vals, ecdf_vals, line_width=3, line_color=colors[i],
        line_dash='dotted', legend_label=f'Exhaustive search')
    p.line(sorted_knn_vals, ecdf_vals, line_width=2,
           line_color=colors[i], legend_label=f'{selection} selected kNN')

p.xaxis.axis_label = r'$$\text{KL Divergence to Nearest Neighbor Model}$$'
p.yaxis.axis_label = r'$$\text{Empirical CDF}$$'
p = dpf.format_fig_fonts(p, font_size=14)
p.legend.location = 'bottom_right'
p.legend.click_policy = "hide"
show(p)